In [1]:
from pathlib import Path
import pandas as pd, numpy as np

RAW = Path("../data/raw")
frames = []
for f in RAW.rglob("*.csv"):
    sensor, room = f.stem.lower().split("_room_")
    d = pd.read_csv(f).rename(columns={f"room_{room}": "value"})
    d["sensor"], d["room"] = sensor, int(room)
    frames.append(d)
raw = pd.concat(frames, ignore_index=True)
raw["minute"] = (pd.to_timedelta(raw["Time"]).dt.total_seconds() // 60).astype(int)
print(raw.shape, raw.DayId.nunique())

(1087315, 9) 44


In [2]:
days = sorted(raw.DayId.unique())
grid = pd.MultiIndex.from_product([days, range(1440)], names=["DayId", "minute"])
day_info = raw.groupby("DayId").agg(workday=("Workday", "first"), month=("Month", "first"))

def series(sensor, room, how="mean"):
    d = raw[(raw.sensor == sensor) & (raw.room == room)]
    return d.groupby(["DayId", "minute"])["value"].agg(how).reindex(grid)

SLOW = ["co2", "temperature", "humidity", "illuminance", "vav"]
CAMS = ["cam1_fw", "cam1_bw", "cam2_fw", "cam2_bw"]

def build_room(r):
    df = pd.DataFrame(index=grid)
    df["occ"] = series("occupant_count", r).groupby(level=0).ffill().fillna(0)
    for c in CAMS:
        df[c] = series(f"occupant_count_{c}", r, "sum").fillna(0)
    for s in SLOW:   # event-driven sensors: hold last value within the day
        raw_s = series(s, r)
        df[s] = raw_s.groupby(level=0).ffill().groupby(level=0).bfill()
        mins = pd.Series(raw_s.index.get_level_values("minute"), index=raw_s.index)
        last_seen = mins.where(raw_s.notna()).groupby(level=0).ffill()
        df[f"{s}_age"] = (mins - last_seen).fillna(mins)   # minutes since last real reading
    df = df.reset_index()
    df["bin"] = df["minute"] // 5
    agg = {"occ": "mean", **{c: "sum" for c in CAMS},
           **{s: "last" for s in SLOW}, **{f"{s}_age": "last" for s in SLOW}}
    out = df.groupby(["DayId", "bin"]).agg(agg).reset_index()
    out["room"] = r
    return out

data = pd.concat([build_room(r) for r in [1, 2, 3]], ignore_index=True)
data["workday"] = data["DayId"].map(day_info.workday).astype(int)
data["month"] = data["DayId"].map(day_info.month)
data["hour_sin"] = np.sin(2 * np.pi * data["bin"] / 288)
data["hour_cos"] = np.cos(2 * np.pi * data["bin"] / 288)
print(data.shape)
print(data.isna().sum()[lambda s: s > 0])

(38016, 22)
Series([], dtype: int64)


In [3]:
H = 6   # 6 bins x 5 min = predict 30 min ahead
data = data.sort_values(["room", "DayId", "bin"]).reset_index(drop=True)
g = data.groupby(["room", "DayId"])

LAG_COLS = ["occ", "co2", "temperature", "humidity", "illuminance", "vav"]
for col in LAG_COLS:
    for lag in [1, 2, 3, 6]:
        data[f"{col}_lag{lag}"] = g[col].shift(lag)
for col in ["co2", "temperature", "humidity"]:
    data[f"{col}_diff"] = g[col].diff()

# door traffic over the last 15 minutes
for c in CAMS:
    data[f"{c}_last15"] = g[c].transform(lambda s: s.rolling(3, min_periods=1).sum())

data["target"] = g["occ"].shift(-H)
data = data.dropna(subset=["target", "occ_lag6", "co2_lag6", "co2_diff"]).reset_index(drop=True)
print(data.shape)

(36432, 54)


In [4]:
cap = {1: 84, 2: 32, 3: 32}
event_days = {(2, 17), (3, 17), (3, 42), (3, 13)}   # over capacity but CO2-supported

data["capacity"] = data["room"].map(cap)
data["event_day"] = [int((r, d) in event_days) for r, d in zip(data["room"], data["DayId"])]

# clip camera-noise occupancy to capacity, leave event days untouched
noise = data["event_day"] == 0
for col in ["occ", "target"] + [f"occ_lag{l}" for l in [1, 2, 3, 6]]:
    data.loc[noise, col] = np.minimum(data.loc[noise, col], data.loc[noise, "capacity"])

# cap extreme single-bin camera values
for c in CAMS + [f"{c}_last15" for c in CAMS]:
    data[c] = data[c].clip(upper=data[c].quantile(.999))

# CO2 floor (outdoor background ~400 ppm, small allowance for sensor error)
for c in ["co2"] + [f"co2_lag{l}" for l in [1, 2, 3, 6]]:
    data[c] = data[c].clip(lower=380)

# room-comparable features
data["occ_ratio"] = data["occ"] / data["capacity"]
data["is_lecture"] = (data["room"] == 1).astype(int)

print(data.groupby("room")[["occ", "target"]].max())
print(data.groupby("room")["event_day"].mean().round(3))

       occ  target
room              
1     78.8    78.8
2     63.4    63.4
3     73.0    73.0
room
1    0.000
2    0.023
3    0.068
Name: event_day, dtype: float64


In [5]:
print(data.shape, "| NaNs:", int(data.isna().sum().sum()))
print(data.groupby("room")["target"].describe().round(2))
print(data.groupby("room")["DayId"].nunique())
data[["occ", "co2", "temperature", "humidity", "illuminance", "vav"]].describe().round(1)

(36432, 58) | NaNs: 0
        count  mean    std  min  25%  50%  75%   max
room                                                
1     12144.0  8.54  16.44  0.0  0.0  0.2  5.0  78.8
2     12144.0  4.25   6.64  0.0  0.0  1.0  5.6  63.4
3     12144.0  3.91   7.29  0.0  0.0  0.4  5.0  73.0
room
1    44
2    44
3    44
Name: DayId, dtype: int64


,occ,co2,temperature,humidity,illuminance,vav
count,36432.0,36432.0,36432.0,36432.0,36432.0,36432.0
mean,5.6,541.5,22.6,41.5,131.7,24.1
std,11.3,105.8,1.4,10.1,188.9,27.9
min,0.0,405.8,16.3,22.7,0.0,0.0
25%,0.0,462.7,21.5,33.7,0.0,0.0
50%,1.0,515.8,22.2,41.8,49.0,19.6
75%,5.0,580.8,23.4,47.8,218.1,45.1
max,78.8,1207.7,28.4,75.7,2064.6,100.0


In [6]:
data.to_csv("../data/processed/model_table_5min.csv", index=False)

## Cleaning summary

**Output:** `data/processed/model_table_5min.csv` (36,432 rows x 58 columns; 12,144 rows per room; 44 days in each room; 0 missing values).

### Steps
1. **Load:** all 30 raw CSVs (10 sensor types x 3 rooms, `original` version) combined into one table; time of day converted to minutes.
2. **Common 5-minute grid** per room and per day:
   - occupancy: mean per bin
   - camera entry/exit flows: summed per bin
   - CO2, temperature, humidity, illuminance, VAV: last value carried forward within the day (the start of a day is filled with that day's first reading)
   - `*_age` columns record the minutes since each sensor's last real reading, so stale filled values are visible to the model
3. **Features:**
   - lags of 5, 10, 15 and 30 min for occupancy, CO2, temperature, humidity, illuminance and VAV
   - recent changes (`_diff`) for CO2, temperature and humidity
   - door traffic over the last 15 min (`*_last15`)
   - time of day (sin/cos), workday flag, month
4. **Target:** occupant count 30 minutes ahead (`H = 6` bins), shifted within each room-day so nothing crosses day boundaries. Rows without a full lag history or a future target are dropped.
5. **Outlier handling:**
   - occupancy (and lags/target) clipped to room capacity (84 / 32 / 32), except on event days
   - camera flow values capped at the 99.9th percentile
   - CO2 floored at 380 ppm
6. **Event days:** four room-days exceed capacity but are supported by CO2 rises (Room 2 day 17; Room 3 days 17, 42, 13). They are kept unclipped and flagged with `event_day`. This flag is for evaluation only and must not be used as a model input.
7. **Room features:** `capacity`, `occ_ratio` (occ / capacity), `is_lecture`.

### Checks
- Row count matches expectation: 44 days x 276 bins x 3 rooms = 36,432.
- Max occupancy: Room 1 stays under its 84 seats; Rooms 2 and 3 exceed 32 only on event days.
- All feature ranges are plausible (CO2 406-1208 ppm, temperature 16-28 C, VAV 0-100%).

### Assumptions and caveats
- Temperature, humidity and VAV are event-driven, so forward-filling assumes the value held until the next reading. Some gaps are 8-13 hours long.
- For VAV (first reading around 07:30), the overnight state is assumed to equal the first reading of the day.
- Occupancy is zero for much of the time (median 0-1 person), so evaluation must not rely on overall error alone.
- Because the data is 44 separate days, train/test splits must be grouped by `DayId`.